In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from datetime import datetime, timedelta
import random

volume_path = "/Volumes/ecommerce/bronze/source_files"

# Shared key pools to keep datasets related
NUM_CUSTOMERS = 50000
NUM_PRODUCTS = 50
customer_ids = [f"CUST-{i:05d}" for i in range(1, NUM_CUSTOMERS + 1)]
product_ids = [f"PROD-{i:03d}" for i in range(1, NUM_PRODUCTS + 1)]

In [0]:
# ORDERS DATA (related to customers + products via shared key pools)

order_data = []
countries = ["DE", "FR", "IT", "ES", "CH"]
for i in range(1, 5001):
    order_data.append({
        "order_id": f"ORD-{i:06d}",
        "customer_id": random.choice(customer_ids),
        "product_id": random.choice(product_ids),
        "country": random.choice(countries),
        "quantity": random.randint(1, 10),
        "unit_price": random.uniform(9, 2299),
        "order_status": random.choice(["completed", "pending", "shipped", "cancelled", "returned"]),
        "order_date": (datetime(2024, 1, 1) + timedelta(days=random.randint(0, 365))).strftime("%Y-%m-%d"),
        "payment_method": random.choice(["credit_card", "debit_card", "upi", "wallet", "cod"]),
        "shipping_city": random.choice([
            "Mumbai", "Delhi", "Bangalore", "Chennai", "Hyderabad",
            "Pune", "Kolkata", "Ahmedabad", "Jaipur", "Lucknow"
        ])
    })

orders_df = spark.createDataFrame(order_data)
orders_df.coalesce(3).write.mode("overwrite").option("header", "true").csv(f"{volume_path}/orders/")

print(f"Wrote {orders_df.count()} orders to {volume_path}/orders/")

In [0]:
customer_data = []
for i, cust_id in enumerate(customer_ids, start=1):
    customer_data.append({
        "customer_id": cust_id,
        "customer_name": f"Customer_{i}",
        "email": f"customer{i}@example.com",
        "signup_date": (datetime(2022, 1, 1) + timedelta(days=random.randint(0, 730))).strftime("%Y-%m-%d"),
        "tier": random.choice(["bronze", "silver", "gold", "platinum"]),
        "city": random.choice(["Mumbai", "Delhi", "Bangalore", "Chennai"]),
        "is_active": random.choice([True, True, True, False])  # 75% active
    })

customers_df = spark.createDataFrame(customer_data)
customers_df.coalesce(1).write.mode("overwrite").option("header", "true").csv(f"{volume_path}/customers/")

print(f"Wrote {customers_df.count()} customers to {volume_path}/customers/")

In [0]:
# PRODUCT CDC EVENTS (change feed for product catalogue dimension)
# Simulates INSERT / UPDATE / DELETE operations on the product catalogue
# Written to products/ — consumed by bronze ingestion as a CDC streaming source
import builtins

categories = ["Electronics", "Clothing", "Kitchen", "Sports", "Books", "Beauty", "Toys"]
brands = ["A", "B", "C", "D", "E"]

cdc_events = []
for i in range(1, 301):
    prod_id = random.choice(product_ids)
    op = random.choice(["INSERT", "UPDATE", "UPDATE", "UPDATE", "DELETE"])
    base_price = builtins.round(random.uniform(9.99, 1999.99), 2)
    cdc_events.append({
        "cdc_timestamp": (datetime(2024, 6, 1) + timedelta(hours=random.randint(0, 8760))).strftime("%Y-%m-%d %H:%M:%S"),
        "operation": op,
        "product_id": prod_id,
        "product_name": f"Product_{prod_id.split('-')[1]}_v2" if op == "UPDATE" else f"Product_{prod_id.split('-')[1]}",
        "category": random.choice(categories),
        "brand": random.choice(brands),
        "unit_price": builtins.round(base_price * random.uniform(0.85, 1.15), 2) if op == "UPDATE" else base_price,
        "stock_quantity": 0 if op == "DELETE" else random.randint(0, 500),
        "is_active": op != "DELETE"
    })

cdc_df = spark.createDataFrame(cdc_events)
cdc_df.coalesce(1).write.mode("overwrite").option("header", "true").csv(f"{volume_path}/products/")

print(f"Wrote {len(cdc_events)} product CDC events to {volume_path}/products/")